# 全遺伝子から創薬標的のトップ500ランキングを作る

HGNC のタンパク質コード遺伝子（約19,300個）から出発し、軽い処理 → 重い処理の順に絞り込んで、疾患ごとに**トップ500のランキング**を作ります。これまでのノートブック（03〜06）で検証した方法を、処理の重さに合わせて並べたものです。

| 段階 | 対象 | 方法 | 残す数 | 時間の目安 |
|---|---|---|---|---|
| 0. 準備（全疾患で共通、1回だけ） | 全遺伝子 | 全遺伝子の機能情報（UniProt）を取得し、疾患に触れる記述を除く | ― | 約10分 |
| 1. ふるい | 全遺伝子 | リランカー（Qwen3-Reranker-0.6B）が疾患の説明と機能説明を組にして読み、治療・症状の改善につながりうるかを採点 | 5,000 | 約1時間 |
| 2. 粗い採点 | 5,000 | M3s を1順だけで聞き、順序の偏りを補正する | 2,000 | 約85分 |
| 3. 精密な採点 | 2,000 | 06 の点数（M3s ＋ 機能情報つき M3sF を両順）＝**最終順位** | 500 | 約3.3時間 |
| 4. 上位の並べ直し | 上位200 | 5択 × スイス式 × Bradley-Terry（確認用。順位は変えない） | ― | 約20分 |
| 5. 理由のラベル | 500 | H3・R1・B1 を両順で聞き、500件の中で特に強い条件をラベルにする | ― | 約25分 |
| 6. 裏付けの注釈（順位付けの後） | 500 | Open Targets（臨床段階は ChEMBL 由来）で既知か新規かを表示 | ― | 数分 |

件数・時間は `config/pipeline07.yaml` で変えられます（時間は txgemma-9b Q6_K、Mac での目安）。生成 LLM は、Mac では txgemma-9b（GGUF）、Linux サーバーでは MedGemma-27B（Hugging Face）を自動で使い分けます（`model.use`）。

## 設定と疾患の追加
- **すべての設定は YAML** です。`config/pipeline07.yaml`（段階ごとの件数・モデル・ラウンド数など）と `config/diseases.yaml`（疾患の定義）。
- **新しい疾患**は `config/diseases.yaml` に1ブロック足すだけです。入力は**病名（name）と疾患の情報（info：直したい症状と、その原因の臓器・細胞・機能の異常の箇条書き）の2つだけ**です。症状の箇条書きには**分子名・遺伝子名・薬の名前を書かない**ルールです（書くと答えを教えることになる）。このノートブックは、箇条書きに遺伝子記号が含まれていないかを自動で確認します。
- 遺伝子のリストは疾患によらず全遺伝子なので、疾患ごとの遺伝子リストの準備は不要です。

## 実行のしかた
- 上から順にセルを実行します。**1回の実行ごとに日付フォルダ** `outputs/genome/<疾患>/<年月日-時分>/` を作り、各段階の結果と条件をまとめて保存します：
  - `conditions/`：実際に使った設定（`config_used.yaml`）、疾患の定義（`disease.yaml`）、モデルに渡した前置きと質問文（`prompts.txt`）
  - `run_log.yaml`：段階ごとの終了時刻、かかった秒数、入力件数・残した件数、主な値（順序の偏り δ など）
  - `stage1.csv` 〜 `stage6.csv`、`top500.csv`、`evaluation.csv`、`top500_charts.html`
- **途中で止まったら** `RUN_ID = "latest"`（または日付フォルダ名）にして上から実行し直すと、そのフォルダで続きから再開します（段階2・3・5は `checkpoint_every` 件ごとに保存）。
- 数時間かかるので、夜間に回すならコマンドライン版（中身は同じ）が便利です。Mac のスリープを防ぐため `caffeinate -i` を付けます：
  `caffeinate -i python scripts/genome_pipeline.py --disease ra`
- 試運転は、下の `OVERRIDES` に小さい件数を入れます。
- 必要なライブラリ：`torch` と `transformers`（段階1のリランカーと、サーバーの生成 LLM）、`llama-cpp-python`（Mac の txgemma）、`plotly`、`pandas`、`pyyaml`。

## 注意
- 段階4の並べ直しは、これまでの検証で主な指標を上げなかったので**順位には使いません**。段階3と段階4の両方で50位以内なら「A」の段にします（確からしさの目安）。
- p（Yes の確率）はダミーでも高めに出るので、絶対基準（0.5 以上なら標的など）には使わず**順位で判断**してください。
- 段階6（Open Targets）は順位を付けたあとの注釈と評価にだけ使います。先に使うと答えを見ることになります。
- 100遺伝子のセットでの検証値（06 の点数：候補 vs ダミー（知名度をそろえる）0.902、既知 vs その他 0.872）は、紛らわしいダミーを混ぜた小さな集合での値です。全遺伝子では、**上位 K 件に既知の標的がどれだけ入るか（recall@K）と、ランダムに選んだ場合の何倍か（濃縮率）**で評価します（最後のセル）。

## 点数の考え方

### 0. 段階1の点数（リランカー）
リランカーに「疾患の説明（Query）」と「遺伝子の説明（Document）」を組で読ませ、「治療・症状の改善につながりうるか」を yes / no で答える直前の確率を読みます。点数は **log P(yes) − log P(no)**（下の対数オッズと同じ考え方）。段階1はふるいなので、この点数は**残すかどうかにだけ使い**、最終順位には使いません。

### 1. Yes の確率 → 対数オッズ
モデルに「3つの条件のどれかに当てはまるか」を Yes/No で聞き、答えを生成させる代わりに、**次の語が Yes になる確率 p と No になる確率を読みます**（Yes・yes・YES などの綴り違いは合算し、Yes と No の2つで合計1になるよう正規化）。
点数には p そのものではなく **対数オッズ log(p / (1 − p))** を使います。p は 0.99 と 0.999 のように1の近くで差が詰まって同点が増えますが、対数オッズなら 4.6 と 6.9 のように差が残り、足したり平均したりしやすいからです。0 が p = 0.5（五分五分）、正なら Yes 寄り、負なら No 寄りです。

### 2. 両順の平均（順序の偏りを消す）
前置きの最後の指示を「Yes or No で答えよ」にするか「No or Yes で答えよ」にするかで、同じ遺伝子でも p が最大 0.3 ほど変わります（先に書かれた方を選びやすい癖）。そこで**両方の順で聞いて対数オッズを平均**します。
段階2では時間を半分にするため「Yes or No」だけで聞き、無作為に選んだ一部の遺伝子で両順の差 δ（平均）を測って、全体を `yes_first − δ/2` と補正します（両順の平均に相当）。

### 3. 最終順位の点数（段階3）＝ M3s と M3sF の平均
- **M3s**：遺伝子名だけを見せて聞く（モデルの記憶で判断）。
- **M3sF**：遺伝子名の後に機能情報（UniProt の機能説明・GO・Reactome の3行）を足して聞く（書かれた事実で判断）。
- **score = (M3s + M3sF) / 2**（どちらも両順の対数オッズの平均）。
検証では、機能情報だけだと「機能欄が治療の仕組みと合う遺伝子（GH1・CYP17A1 など）は上がるが、合わない標的（FOLH1 など）は下がる」と疾患によって得失があり、**2つを平均すると互いの弱点を補って主な指標がすべて最良**になりました（100遺伝子のセットで、知名度をそろえた候補 vs ダミーの AUC 0.902、M3s だけなら 0.863）。

### 4. 段階4の強さ（Bradley-Terry）
5遺伝子＋「該当なし」から1つ選ばせ、遺伝子 i が選ばれる確率を **π_i ÷（グループ内の π の合計）** と置いて、全ラウンドの結果に最もよく合う強さ π を推定します（Luce の選択モデル。2択の Bradley-Terry を多者択一に広げたもの）。強い相手に勝つほど π が大きくなるので、相手の強さを考慮した順位になります。スイス式（強さの近い遺伝子どうしで組む）でこの補正が効きます。段階4は**確認用**で、最終順位は変えません。

### 5. 確からしさの段
- **A**：段階3で 1〜50 位、かつ段階4でも 50 位以内（2つの聞き方で一致）。
- **B**：〜200 位（A 以外）。**C**：〜500 位。
上位ほど2つの聞き方の結果が一致して安定し、200 位以降は点数の差が小さいので、細かい順位より段で見てください。

### 6. 理由のラベル（段階5）
H3（直接効く）・R1（代償経路）・B1（機構が一致）を別々に聞き、各条件の対数オッズを500件の中で標準化（平均0・標準偏差1）して、**平均より 0.5 標準偏差以上高い条件**をラベルにします。上位の遺伝子は H3・R1 がほぼ全員 Yes、B1 はほぼ全員 No なので、「500件の中で相対的に強いか」で見ないと区別が付かないためです。

### 7. 評価（最後のセル。順位付けには使わない）
- **recall@K**：正解（既知の標的）のうち、上位 K 件に入った割合。
- **濃縮率**：上位 K 件に正解が入る割合 ÷（正解の数 ÷ 全遺伝子数）。ランダムに K 件選んだ場合の何倍か。
- 正解は、`evaluation.known_files` の既知標的（これまでの検証で使った一覧。ある疾患だけ）と、Open Targets で PHASE_2 以上の薬がある標的の2種類です。

## 設定を読む

### このセルがすること：YAML を読み、実行する疾患を選ぶ
- `OVERRIDES`：YAML を書き換えずに一時的に設定を変える（試運転用）。例 `["stage1.keep=300", "stage2.keep=100", "stage3.keep=50", "stage4.top=20", "paths.out_dir=outputs/genome_trial"]`
- `DISEASE`：実行する疾患の病名（`config/diseases.yaml` の name。空なら `config/pipeline07.yaml` の `run_diseases` の先頭）。
- `RUN_ID`：結果を入れる日付フォルダ。空なら新しく作る（年月日-時分）。`"latest"` なら最新のフォルダ、フォルダ名を書けばそのフォルダで続きから再開。
- 条件（設定・疾患の定義）を `conditions/` に、実行の記録を `run_log.yaml` に保存します。

In [ ]:
import os, re, sys, json, math, glob, time, random, urllib.request
import numpy as np
import pandas as pd
import yaml
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
ROOT = os.path.abspath("..")

CONFIG = os.path.join(ROOT, "config", "pipeline07.yaml")
OVERRIDES = []
DISEASE = ""                          # 病名（例 "schizophrenia"）。空なら run_diseases の先頭

RUN_ID = ""                           # "" = 新しい日付フォルダ / "latest" = 最新のフォルダで再開 / "20260926-1530" など
# --- 設定ファイル（YAML）を読む ---
cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
for o in OVERRIDES:                                   # "stage1.keep=300" → cfg["stage1"]["keep"] = 300
    k, v = o.split("=", 1); node = cfg; parts = k.split(".")
    for p in parts[:-1]: node = node[p]
    node[parts[-1]] = yaml.safe_load(v)
# --- 疾患の定義を読み、実行する疾患を選ぶ ---
DISEASES = {d["name"]: d for d in yaml.safe_load(open(os.path.join(ROOT, cfg["diseases_file"]), encoding="utf-8"))["diseases"]}
D = DISEASES[DISEASE or cfg["run_diseases"][0]]                                  # 選んだ疾患（name と info の2つ）
DISEASE_KEY = re.sub(r"[^a-z0-9]+", "_", D["name"].lower()).strip("_")          # フォルダ名：病名の空白などを _ に（例 rheumatoid_arthritis）
P = {k: os.path.join(ROOT, os.path.expanduser(v)) for k, v in cfg["paths"].items()}       # 設定のパスを絶対パスに
# --- 結果を入れる日付フォルダを決める（新規 / 最新 / 指定） ---
BASE = os.path.join(P["out_dir"], DISEASE_KEY); os.makedirs(BASE, exist_ok=True)
if RUN_ID == "latest": RUN_ID = sorted(d for d in os.listdir(BASE) if re.fullmatch(r"\d{8}-\d{4}", d))[-1]
RUN_ID = RUN_ID or time.strftime("%Y%m%d-%H%M")
OUT = os.path.join(BASE, RUN_ID); os.makedirs(os.path.join(OUT, "conditions"), exist_ok=True)
# --- 条件を保存する：実際に使った設定（OVERRIDES 反映後）と疾患の定義 ---
yaml.safe_dump({**cfg, "_overrides": OVERRIDES, "_disease_key": DISEASE_KEY, "_run_id": RUN_ID},
               open(os.path.join(OUT, "conditions", "config_used.yaml"), "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
yaml.safe_dump(D, open(os.path.join(OUT, "conditions", "disease.yaml"), "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
# --- 実行の記録（段階ごとに追記して保存する。再開したときは再開の時刻を足す） ---
RUNLOG_PATH = os.path.join(OUT, "run_log.yaml")
if os.path.exists(RUNLOG_PATH):
    RUNLOG = yaml.safe_load(open(RUNLOG_PATH, encoding="utf-8")); RUNLOG.setdefault("resumed", []).append(time.strftime("%Y-%m-%d %H:%M:%S"))
else:
    RUNLOG = {"run_id": RUN_ID, "disease": DISEASE_KEY, "started": time.strftime("%Y-%m-%d %H:%M:%S")}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
# --- この計算機の種類（auto：Mac なら mac、Linux なら server）。段階1のリランカーと生成 LLM の設定を選ぶのに使う ---
MACHINE = cfg["model"]["use"] if cfg["model"]["use"] != "auto" else ("mac" if sys.platform == "darwin" else "server")
pick = lambda v: v[MACHINE] if isinstance(v, dict) else v                   # {mac: …, server: …} の形なら、この計算機の値を取る
# 臨床段階の並び（段階6で「最も進んだ段階」を決めるのに使う。左ほど早い）
STAGE_ORDER = ["PRECLINICAL", "IND", "EARLY_PHASE_1", "PHASE_1", "PHASE_1_2", "PHASE_2", "PHASE_2_3", "PHASE_3", "PREAPPROVAL", "APPROVAL"]
print("登録疾患:", list(DISEASES))
print(f"選択: {D['name']} | 出力先: {OUT} | 計算機: {MACHINE}")
for b in D["info"]: print("  -", b)
print({k: cfg[k] for k in ("stage1", "stage2", "stage3", "stage4")})

### このセルがすること：全遺伝子のリストを読み、症状の箇条書きに遺伝子記号が含まれていないかを確認する
- プロンプトに入れる遺伝子名は「記号 (タンパク質名)」です。

In [ ]:
# --- 全遺伝子（HGNC のタンパク質コード遺伝子）を読む ---
U = pd.read_csv(P["gene_universe"], sep="\t", dtype=str).fillna("").drop_duplicates("symbol").reset_index(drop=True)
U["acc"] = U["uniprot_ids"].str.split("|").str[0]                                    # UniProt の代表 ID
U["gene_label"] = [f"{s} ({(p or n).split('|')[0]})" if (p or n) else s for s, p, n in zip(U["symbol"], U["protein_name_uniprot"], U["gene_name"])]
G = U.set_index("symbol")                                                # 記号で引けるようにする
# --- 症状の箇条書きに遺伝子記号（大文字3字以上の語で、HGNC に存在するもの）が紛れていないか確認 ---
hits = sorted({w for b in D["info"] for w in re.findall(r"\b[A-Z][A-Z0-9]{2,}\b", b) if w in G.index})
print(f"全遺伝子 {len(U)} 個 |", "症状の箇条書きに遺伝子記号なし（OK）" if not hits else f"[警告] 箇条書きに遺伝子記号が含まれます: {hits}")

### このセルがすること：質問文と前置きの注意書きを定義する（文面は検証スクリプトからそのまま）
- Yes/No の質問は `scripts/yesno_question_variants.py`、5択の質問は `scripts/rank_variants.py` の文面です。検証した文面を1文字も変えずに使います。
- YAML で別の質問 ID を指定するときは、ここに文面を足してください（変えたら検証し直してください）。

In [ ]:
YN_QUESTIONS = {
 "M3s": "Consider these three criteria:\n(a) Inhibiting or activating {gene} could plausibly treat {disease} or improve at least one of the symptoms listed above.\n(b) Even outside the causal pathway, modulating {gene} could counteract the abnormal process described above through a parallel or opposing pathway.\n(c) {gene}'s own substrate, ligand, pathway, cell type or circuit precisely matches the mechanism of {disease} described above (a similar but distinct role, even in the same gene family, does not count).\nDoes this gene meet at least one of these criteria?",
 "H3": "Can a hypothesis be formulated that a therapeutic drug targeting the gene {gene} would treat {disease} or the symptoms of {disease}?",
 "R1": "Even if this gene is not part of the pathway that causes {disease}, could activating or inhibiting it counteract or compensate for the abnormal process described above, for example through a parallel or opposing pathway in the same cells?",
 "B1": "Does this gene's own specific role — its substrate, ligand, signalling pathway, cell type or circuit — match the mechanism described above precisely, rather than a related but distinct one (e.g. a different molecule, cell type, tissue or subcellular compartment)? Answer No for a similar-sounding but distinct role, even in the same gene family."
}
CHOICE_QUESTIONS = {
 "M2r": "Consider these three criteria:\n(a) A hypothesis can be constructed that inhibiting or activating the gene would treat {disease} or improve at least one of the symptoms listed above.\n(b) Even if the gene is not part of the pathway that causes {disease}, activating or inhibiting it could counteract or compensate for the abnormal process described above, for example through a parallel or opposing pathway in the same cells.\n(c) The gene's own specific role — its substrate, ligand, signalling pathway, cell type or circuit — matches the mechanism described above precisely, rather than a related but distinct one (e.g. a different molecule, cell type, tissue or subcellular compartment); a similar-sounding but distinct role, even in the same gene family, does not count.\nWhich numbered gene above best meets at least one of these criteria?"
}
YN_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Answer Yes only when there is clear evidence or a clear mechanistic link; otherwise answer No. Membership in the same gene family, superfamily or protein class as a true disease gene (e.g. being another member of the same transporter, channel, receptor or enzyme family) is NOT sufficient evidence by itself. Judge each gene on whether ITS OWN specific substrate, ligand, cargo or interaction partner matches the mechanism described above, not on family resemblance alone.\n'
CHOICE_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.\n'
GROUP_SIZE = 5

# --- 前置き（全遺伝子で共通の部分）を組み立てる：役割 → 注意書き → 病名 → 症状の箇条書き → 答え方の指示 ---
BULLETS = "\n".join(f"- {b}" for b in D["info"])
YN_PREFIX = {order: ("You are an expert in drug discovery and human disease biology.\n" + YN_NOTE +
                     f"Disease: {D['name']}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{BULLETS}\n" +
                     ("Answer each question with Yes or No." if order == "yes_first" else "Answer each question with No or Yes.") + "\n\n")
             for order in ("yes_first", "no_first")}
CHOICE_PREFIX = ("You are an expert in drug discovery and human disease biology. You will be shown a numbered list of "
                 f"{GROUP_SIZE} candidate genes for one disease, and asked which ONE number is the best answer to a question.\n" + CHOICE_NOTE +
                 f"Disease: {D['name']}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{BULLETS}\n"
                 f"Answer with a single number from 1 to {GROUP_SIZE + 1} only. No words, no explanation.\n\n")
# --- 条件として、前置きと質問文をそのまま保存する ---
with open(os.path.join(OUT, "conditions", "prompts.txt"), "w", encoding="utf-8") as f:
    for order in YN_PREFIX: f.write(f"===== Yes/No の前置き（{order}）\n{YN_PREFIX[order]}\n")
    f.write(f"===== 5択の前置き\n{CHOICE_PREFIX}\n")
    for k, v in {**YN_QUESTIONS, **CHOICE_QUESTIONS}.items(): f.write(f"===== 質問 {k}\n{v}\n\n")
# 実際にモデルに渡すプロンプトの例（TNF）を表示して確認する
print(YN_PREFIX["yes_first"] + f"Gene: {G.at['TNF', 'gene_label']}\n" + f"M3s. {YN_QUESTIONS['M3s'].format(disease=D['name'], gene='TNF')} Answer:")

## 図の準備

### このセルがすること：図を保存するフォルダ `charts/` を作り、図に既知の標的の位置を示すための一覧を読む
- 各段階の図は `charts/NN_名前.html` に保存し、最後に一覧 `charts/index.html` を作ります。グラフの描画ライブラリ（plotly.js）は `charts/plotly.min.js` を1つだけ置いて全部の図から参照するので、ネットにつながっていなくても開け、1枚あたりのファイルも軽くなります。
- 既知の標的（`config/pipeline07.yaml` の `evaluation.known_files` にこの病名があれば、その一覧）は、図で「どこにいるか」を示すためだけに使います。順位付けには使いません。無ければ目印なしで描きます。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
CHARTS = os.path.join(OUT, "charts"); os.makedirs(CHARTS, exist_ok=True)
CHART_TITLES = {}                                                          # ファイル名 → 図の題名（最後に一覧ページを作るのに使う）
KNOWN = set()
KNOWN_FILE = cfg.get("evaluation", {}).get("known_files", {}).get(D["name"])   # 答え合わせ用の一覧（pipeline07.yaml の evaluation。無くてもよい）
if KNOWN_FILE and os.path.exists(os.path.join(ROOT, KNOWN_FILE)):
    KNOWN = set(pd.read_csv(os.path.join(ROOT, KNOWN_FILE), sep="\t")["symbol"])   # 図で位置を示すだけ（順位付けには使わない）
PLOT = dict(template="plotly_white", font=dict(family="Hiragino Sans, Noto Sans JP, sans-serif", size=12))
print("図の保存先:", CHARTS, "| 既知の標的（図の目印用）:", len(KNOWN))

## 段階0：準備（全疾患で共通、1回だけ）

### このセルがすること：全遺伝子の機能情報を UniProt から取得して保存する
- 取得するのは Function（最初の2文、最大320文字）、GO の分子機能2つ・生物学的プロセス4つ、Reactome の経路名3つです。「Involvement in disease」（どの病気に関わるか）の欄は取得しません。
- ここでは疾患の語は除きません（全疾患で共通に使うため）。疾患ごとの除去は次のセルで行います。取得済みの分は飛ばすので、2回目以降はすぐ終わります。

In [ ]:
# --- 取得済みの機能情報を読み、まだ無い遺伝子（UniProt ID）を洗い出す ---
cache = json.load(open(P["uniprot_function"])) if os.path.exists(P["uniprot_function"]) else {}
todo = sorted({a for a in U["acc"] if a and a not in cache})
print(f"機能情報: 全 {len(U)} 遺伝子のうち取得済み {len(U) - len(todo)}、残り {len(todo)}")
B = cfg["stage0"]["uniprot_batch"]
# --- UniProt に B 件ずつ問い合わせる ---
for i in range(0, len(todo), B):
    url = ("https://rest.uniprot.org/uniprotkb/accessions?accessions=" + ",".join(todo[i:i + B]) +
           "&fields=accession,cc_function,go_p,go_f,xref_reactome&format=json")
    with urllib.request.urlopen(url, timeout=60) as r:
        entries = json.load(r)["results"]
    for e in entries:
        # Function：文献・証拠コードを除き、文に分けて保存する（使うときに最初の2文を取る）
        func = " ".join(t["value"] for c in e.get("comments", []) if c.get("commentType") == "FUNCTION" for t in c.get("texts", []))
        func = re.sub(r"\(PubMed:[^)]*\)|\{ECO:[^}]*\}", "", func)
        sents = [s.strip() for s in re.split(r"(?<=[.;])\s+", func) if s.strip()]
        # GO（分子機能・生物学的プロセス）と Reactome の名前
        go_p, go_f, rea = [], [], []
        for x in e.get("uniProtKBCrossReferences", []):
            props = {p["key"]: p["value"] for p in x.get("properties", [])}
            name = props.get("GoTerm", props.get("PathwayName", ""))
            if x["database"] == "GO" and name.startswith("P:"): go_p.append(name[2:])
            if x["database"] == "GO" and name.startswith("F:"): go_f.append(name[2:])
            if x["database"] == "Reactome": rea.append(name)
        cache[e["primaryAccession"]] = {"function": sents, "go_f": go_f, "go_p": go_p, "reactome": sorted(set(rea), key=len)}
    for a in todo[i:i + B]:                                                  # UniProt に無かったもの
        cache.setdefault(a, {"function": [], "go_f": [], "go_p": [], "reactome": []})
    if (i // B) % 20 == 0 or i + B >= len(todo):
        json.dump(cache, open(P["uniprot_function"], "w"), ensure_ascii=False); print(f"  UniProt {min(i + B, len(todo))}/{len(todo)}")
    time.sleep(0.2)
RUNLOG["stage0_function"] = {"cache": P["uniprot_function"], "entries": len(cache), "no_function": sum(not v["function"] for v in cache.values())}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(f"機能情報 OK：{len(cache)} 件 | Function の記載なし {sum(not v['function'] for v in cache.values())} 件")

### このセルがすること：この疾患の病名を含む文・項目を除き、遺伝子ごとの機能情報を作る
- 除くのは `config/diseases.yaml` の、この疾患の `name`（病名）を含むものだけです（小文字の部分一致）。Function は語を含む**文**を、GO・Reactome は語を含む**名前**を除き、残りから Function 2文（最大320文字）・GO 6つ・Reactome 3つを取ります。
- 答えそのもの（「関節リウマチ患者の …」など）を見せないための処置です。関連する語（炎症・免疫など）は残ります。完全に除くことは目的にしていません。
- ここで作る `FUNC_TEXT`（3行の文章）を、段階1（リランカーの文書）と段階3（M3sF の機能情報）の両方で使います。

In [ ]:
LEAK = [D["name"].lower()]                                                     # 除く語＝この疾患の病名
ok = lambda t: not any(w in t.lower() for w in LEAK)                           # 語を含まなければ True
FUNC_TEXT, n_removed = {}, 0                                                   # 記号 → 「Function: …\nGO: …\nPathways: …」
for s, a in zip(U["symbol"], U["acc"]):
    v = cache.get(a, {"function": [], "go_f": [], "go_p": [], "reactome": []})
    sents = [x for x in v["function"] if ok(x)]
    go_f, go_p, rea = [x for x in v["go_f"] if ok(x)], [x for x in v["go_p"] if ok(x)], [x for x in v["reactome"] if ok(x)]
    n_removed += (len(v["function"]) - len(sents)) + (len(v["go_f"]) + len(v["go_p"]) + len(v["reactome"]) - len(go_f) - len(go_p) - len(rea))
    f = " ".join(sents[:2])
    if len(f) > 320: f = f[:320].rsplit(" ", 1)[0] + "…"
    FUNC_TEXT[s] = (f"Function: {f or 'not described'}\nGO: {'; '.join(go_f[:2] + go_p[:4]) or 'not described'}\n"
                    f"Pathways: {'; '.join(rea[:3]) or 'not described'}\n")
HAS_FUNC = {s: not t.startswith("Function: not described") for s, t in FUNC_TEXT.items()}
print(f"除く語: {LEAK} | 除いた文・項目: {n_removed} 個 | Function の記載あり {sum(HAS_FUNC.values())} / {len(U)}")
print(FUNC_TEXT["DHFR"])

## 段階1：ふるい（リランカー）

### このセルがすること：全遺伝子を、リランカーで「疾患の治療・症状の改善につながりうるか」採点し、上位 `stage1.keep` 件を残す
- **リランカー**は、質問（疾患の説明）と文書（遺伝子の説明）を**組にして一緒に読み**、合っているかを点数にする小さなモデルです。既定は Qwen3-Reranker-0.6B（6億パラメータ）。生成 LLM（txgemma-9b は90億）の十数分の一の大きさなので、全遺伝子でも約1時間で終わります。
- 入力：`<Instruct>`＝指示文（治療・症状の改善につながりうるかを判断せよ）、`<Query>`＝病名＋症状の箇条書き、`<Document>`＝「記号 (タンパク質名).」＋機能情報の3行（前のセルの `FUNC_TEXT`。この疾患の病名を含む文は除いてある）。
- 点数 `score1` ＝ 次の語が yes になる対数確率 − no になる対数確率（0＝五分五分、正なら yes 寄り）。
- 以前の埋め込み（bge-m3 の類似度）より、関節リウマチの既知の標的15個のうち上位2,000件に入る数が 5 → 13 に増えました（`scripts/rerank_stage1.py` で比較）。
- モデル・書式・装置（cuda / mps / cpu）・バッチの大きさは `config/pipeline07.yaml` の `stage1` で変えられます。モデルは計算機ごとで、Mac は Qwen3-Reranker-0.6B、Linux サーバーは Qwen3-Reranker-8B です。
- `stage1.checkpoint_every` 件ごとに `stage1_partial.csv` に保存し、再実行すると続きから再開します。終わったらモデルをメモリから外します（txgemma を読み込む前）。
- 出力 `stage1.csv`：`score1`、`rank1`、`keep1`（残すかどうか）。必要なライブラリ：`torch`、`transformers`。

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
c = cfg["stage1"]; T = c["template"]; RMODEL = pick(c["model"])            # この計算機で使うリランカー
f1 = os.path.join(OUT, "stage1.csv"); f1p = os.path.join(OUT, "stage1_partial.csv")
if os.path.exists(f1):                                                 # 終わっていれば読むだけ（再開したとき）
    s1 = pd.read_csv(f1); print("段階1: 保存済みの結果を読みました")
else:
    # --- 計算に使う装置を決める（auto なら cuda → mps → cpu の順） ---
    DEV = c["device"] if c["device"] != "auto" else ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    DTYPE = torch.float32 if DEV == "cpu" else getattr(torch, pick(c["dtype"]))
    # --- リランカーを読み込む（初回は Hugging Face からダウンロード） ---
    rtok = AutoTokenizer.from_pretrained(RMODEL, padding_side="left")   # 左詰め：各文の最後の位置をそろえる
    rmodel = AutoModelForCausalLM.from_pretrained(RMODEL, dtype=DTYPE).to(DEV).eval()
    YES_ID, NO_ID = rtok.convert_tokens_to_ids(T["yes_token"]), rtok.convert_tokens_to_ids(T["no_token"])
    # --- 質問（疾患の説明）と、全遺伝子の文書（遺伝子の説明）を作る ---
    query = f"{D['name']}. " + " ".join(D["info"])
    docs = {s: f"{G.at[s, 'gene_label']}.\n{FUNC_TEXT[s]}" for s in U["symbol"]}   # 記号 (タンパク質名). ＋機能情報の3行
    # --- 途中まで終わっていれば、その結果を読んで続きから ---
    rows = pd.read_csv(f1p).to_dict("records") if os.path.exists(f1p) else []
    todo = [s for s in U["symbol"] if s not in {r["symbol"] for r in rows}]
    print(f"段階1: {RMODEL}（{DEV}）| 全 {len(U)} 遺伝子のうち採点済み {len(rows)}、残り {len(todo)}")
    t0 = time.time(); BS = pick(c["batch"])
    for i in range(0, len(todo), BS):
        chunk = todo[i:i + BS]
        texts = [T["prefix"] + T["body"].format(instruction=c["instruction"], query=query, document=docs[s]) + T["suffix"] for s in chunk]
        x = rtok(texts, padding=True, truncation=True, max_length=c["max_length"], return_tensors="pt").to(DEV)
        with torch.no_grad():
            lg = rmodel(**x, logits_to_keep=1).logits[:, -1, :].float()  # 最後の位置だけ、次の語の確率を計算する（全位置だとメモリが足りない）
        for s, v in zip(chunk, (lg[:, YES_ID] - lg[:, NO_ID]).cpu().tolist()):   # yes と no のロジットの差＝log P(yes) − log P(no)
            rows.append({"symbol": s, "score1": v})
        if (i + BS) % c["checkpoint_every"] < BS or i + BS >= len(todo):
            pd.DataFrame(rows).to_csv(f1p, index=False)
            done = i + len(chunk); print(f"  {done}/{len(todo)}（{time.time() - t0:.0f}秒、残り約 {(time.time() - t0) / done * (len(todo) - done) / 60:.0f} 分）")
    # --- 点数の高い順に並べ、上位 stage1.keep 件に印を付ける ---
    s1 = pd.DataFrame(rows).sort_values("score1", ascending=False).reset_index(drop=True)
    s1["rank1"] = np.arange(1, len(s1) + 1); s1["keep1"] = s1["rank1"] <= c["keep"]
    s1.to_csv(f1, index=False)
    RUNLOG["stage1"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "seconds": round(time.time() - t0), "model": RMODEL, "device": DEV,
                        "input": len(s1), "kept": int(s1["keep1"].sum())}
    yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    # --- リランカーをメモリから外す（このあと生成 LLM を読み込むため） ---
    del rmodel; gc.collect()
    if DEV == "mps": torch.mps.empty_cache()
    if DEV.startswith("cuda"): torch.cuda.empty_cache()
print(f"段階1: {len(s1)} → {int(s1['keep1'].sum())} 件")
s1.head(15)

### 図1：段階1の点数の分布
### このセルがすること：全遺伝子のリランカーの点数の分布と、残す線（`stage1.keep` 件目）を描き、既知の標的がどこにいるかを示す
- 線より左の既知の標的は、段階1で落ちて生成 LLM に届きません（ふるいの取りこぼし）。

In [ ]:
cut = s1.loc[s1["keep1"], "score1"].min()                                # 残した中で最も低い点数＝ふるいの線
kn = s1[s1["symbol"].isin(KNOWN)]
fig = go.Figure(go.Histogram(x=s1["score1"], nbinsx=80, marker_color="#c9c8c2", name="全遺伝子"))
fig.add_vline(x=cut, line=dict(color="#0b0b0b", dash="dot"), annotation_text=f"上位 {cfg['stage1']['keep']} 件の線")
fig.add_trace(go.Scatter(x=kn["score1"], y=[0] * len(kn), mode="markers+text", text=kn["symbol"], textposition="top center", name="既知の標的",
                         marker=dict(color=np.where(kn["keep1"], "#2a78d6", "#d64545"), size=10, symbol="triangle-up"),
                         customdata=np.c_[kn["symbol"], kn["rank1"]], hovertemplate="<b>%{customdata[0]}</b><br>点数 %{x:.2f}（%{customdata[1]}位）<extra></extra>"))
fig.update_layout(**PLOT, height=420, xaxis_title="リランカーの点数（log P(yes) − log P(no)）", yaxis_title="遺伝子の数",
                  title=f"段階1：リランカーの点数の分布（▲青＝残った既知の標的、▲赤＝ここで落ちた既知の標的：{int((~kn['keep1']).sum())} / {len(kn)}）")
fig.show(); name = "01_stage1_reranker.html"; CHART_TITLES[name] = "段階1：リランカーの点数の分布とふるいの線"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

## 生成 LLM の準備（Mac：txgemma の GGUF／Linux サーバー：MedGemma など Hugging Face のモデル）

### このセルがすること：`config/pipeline07.yaml` の `model` から、この計算機の設定を選んでモデルを読み込み、採点の関数を2つ定義する
- `model.use: auto` なら、Mac では `model.mac`（llama.cpp で GGUF の txgemma-9b）、Linux では `model.server`（transformers で `google/medgemma-27b-it`）を使います。
- `prompt_wrap: none`：書式なしの文章の「… Answer:」の直後で、次の語の確率を読みます（txgemma で検証してきた方式）。
- `prompt_wrap: chat`：文章全体をモデルのチャット書式（ユーザーの発言）で包み、モデルの答えの最初の語の確率を読みます（MedGemma など指示に従うよう調整されたモデル向け）。
- **質問文・前置きは txgemma-9b で検証した文面です。** 別のモデルでは効き方が変わるので、100遺伝子のセット（03・06）で確かめてから本番に使ってください。

### `yes_logodds(label, symbol, qids, order, extra="")` — Yes/No の質問に答えさせる
どんな def か：前置き（`order` ＝ "yes_first" か "no_first"）＋遺伝子行「Gene: 記号 (タンパク質名)」＋`extra`（機能情報。空なら無し）＋質問文を、**質問ごとに独立に**モデルに読ませ、次の語が Yes か No かの確率を読みます（Yes・yes・YES などの綴り違いは合算）。llama.cpp では前置きの処理結果を保存して使い回し、transformers では質問をまとめて1回で計算します。
return：dict（質問 ID → Yes の対数オッズ。±13.8 で頭打ち）。

### `choice_probs(labels, qid)` — 5遺伝子＋「該当なし」から1つ選ばせる
どんな def か：5択用の前置き＋「1. 遺伝子 … 5. 遺伝子 6. None of the above …」＋質問文を読ませ、答えの最初の語が数字 1〜6 のどれかの確率を読み、6つの中で正規化します（`none` では「Answer: 」の末尾の空白の直後）。
return：numpy 配列（長さ6、最後が「該当なし」）。

In [ ]:
# --- この計算機で使うモデルの設定を選ぶ（auto：Mac なら mac、Linux なら server） ---
MODEL_USE = MACHINE                                                        # 設定のセルで決めた計算機の種類
M = cfg["model"][MODEL_USE]; CHAT = M["prompt_wrap"] == "chat"
YES_WORDS, NO_WORDS = ["Yes", " Yes", "yes", " yes", "YES", " YES"], ["No", " No", "no", " no", "NO", " NO"]   # 綴り違いは合算する
CAP = math.log((1 - 1e-6) / 1e-6); clip = lambda x: max(-CAP, min(CAP, x))   # 対数オッズの頭打ち（p を 1e-6〜1−1e-6 に丸めるのと同じ、±13.8）

if M["backend"] == "llama_cpp":
    # ================= Mac：GGUF を llama.cpp で動かす =================
    from llama_cpp import Llama
    assert not CHAT, "llama_cpp では prompt_wrap: none を使ってください"
    MODEL_PATH = M["gguf"]
    if MODEL_PATH == "auto":
        MODEL_PATH = sorted(glob.glob(os.path.join(os.path.expanduser(M["model_dir"]), "**", "*txgemma*.gguf"), recursive=True))[0]
    llm = Llama(model_path=MODEL_PATH, n_ctx=M["n_ctx"], n_gpu_layers=-1, logits_all=False, verbose=False)   # GPU を全層で使う
    tok = lambda text, bos=False: llm.tokenize(text.encode("utf-8"), add_bos=bos, special=bos)
    # --- Yes / No の綴り違いのうち1トークンになるものの番号、数字 1〜6 の番号 ---
    YES_IDS = list(dict.fromkeys(t[0] for t in (tok(s) for s in YES_WORDS) if len(t) == 1))
    NO_IDS = list(dict.fromkeys(t[0] for t in (tok(s) for s in NO_WORDS) if len(t) == 1))
    DIGIT = {n: tok(str(n))[0] for n in range(1, GROUP_SIZE + 2)}
    # --- 前置きを3通り処理して、その時点の状態（KV キャッシュ）を保存する。以後は遺伝子の部分だけ処理すればよい ---
    STATE = {}
    for name, text in (("yes_first", YN_PREFIX["yes_first"]), ("no_first", YN_PREFIX["no_first"]), ("choice", CHOICE_PREFIX)):
        llm.reset(); llm.eval(tok(text, bos=True)); STATE[name] = llm.save_state()
    lse = lambda v: max(v) + math.log(sum(math.exp(x - max(v)) for x in v))   # log(Σ exp)：綴り違いの確率の合算

    def yes_logodds(label, symbol, qids, order, extra=""):
        llm.reset(); llm.load_state(STATE[order])                            # 前置きの状態を復元（前置きは再処理しない）
        llm.eval(tok(f"Gene: {label}\n" + extra)); base = llm.n_tokens       # 遺伝子行（＋機能情報）を処理し、位置を覚える
        out = {}
        for q in qids:
            llm.n_tokens = base                                                   # 遺伝子行の直後へ巻き戻す（質問を独立に聞く）
            llm.eval(tok(f"{q}. {YN_QUESTIONS[q].format(disease=D['name'], gene=symbol)} Answer:"))
            lg = np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
            out[q] = float(clip(lse([lg[i] for i in YES_IDS]) - lse([lg[i] for i in NO_IDS])))   # log(Yes 系) − log(No 系)＝Yes の対数オッズ
        return out

    def choice_probs(labels, qid):
        llm.reset(); llm.load_state(STATE["choice"])                         # 5択用の前置きを復元
        llm.eval(tok("Candidate genes:\n" + "\n".join(f"{i + 1}. {g}" for i, g in enumerate(labels)) + f"\n{len(labels) + 1}. None of the above genes seem clearly relevant\n"))
        llm.eval(tok(f"Question: {CHOICE_QUESTIONS[qid].format(disease=D['name'])} Answer: "))
        lg = np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
        l = np.array([lg[DIGIT[n]] for n in range(1, GROUP_SIZE + 2)]); p = np.exp(l - l.max())   # 数字1〜6だけを取り出して正規化
        return p / p.sum()
    MODEL_NAME = MODEL_PATH

else:
    # ================= Linux サーバー：Hugging Face のモデルを transformers で動かす =================
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForImageTextToText
    SPLIT = M["device"] == "split"                                          # split = 見えている GPU すべてにモデルを分けて載せる
    GDEV = ("cuda" if SPLIT else M["device"]) if M["device"] != "auto" else ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    gtok = AutoTokenizer.from_pretrained(M["name"], padding_side="left")      # 左詰め：各文の最後の位置をそろえる
    try:                                                                     # 文章だけのモデル
        gmodel = AutoModelForCausalLM.from_pretrained(M["name"], dtype=getattr(torch, M["dtype"]), device_map="auto" if SPLIT else None)
    except ValueError:                                                       # 画像も読めるモデル（medgemma-27b-it など）
        gmodel = AutoModelForImageTextToText.from_pretrained(M["name"], dtype=getattr(torch, M["dtype"]), device_map="auto" if SPLIT else None)
    gmodel = (gmodel if SPLIT else gmodel.to(GDEV)).eval()                  # split なら読み込み時に各 GPU へ配置済み
    # --- Yes / No の綴り違いのうち1トークンになるものの番号、数字 1〜6 の番号 ---
    ids1 = lambda words: list(dict.fromkeys(t[0] for t in (gtok.encode(w, add_special_tokens=False) for w in words) if len(t) == 1))
    YES_IDS, NO_IDS = ids1(YES_WORDS), ids1(NO_WORDS)
    DIGIT = {n: gtok.encode(str(n), add_special_tokens=False)[0] for n in range(1, GROUP_SIZE + 2)}
    # --- 文章をモデルに渡す形にする：chat ならユーザーの発言として包み、答えの直前まで。none なら先頭記号＋文章 ---
    wrap = lambda text: (gtok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
                         if CHAT else (gtok.bos_token or "") + text)

    def yes_logodds(label, symbol, qids, order, extra=""):
        texts = [wrap(YN_PREFIX[order] + f"Gene: {label}\n" + extra + f"{q}. {YN_QUESTIONS[q].format(disease=D['name'], gene=symbol)}" + ("" if CHAT else " Answer:"))
                 for q in qids]                                              # 質問ごとに独立の文章（前の答えは持ち込まない）
        x = gtok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(GDEV)
        with torch.no_grad():
            lp = torch.log_softmax(gmodel(**x, logits_to_keep=1).logits[:, -1, :].float(), dim=-1)   # 最後の位置の、次の語の対数確率
        ly, ln = torch.logsumexp(lp[:, YES_IDS], dim=1), torch.logsumexp(lp[:, NO_IDS], dim=1)       # 綴り違いを合算
        return {q: clip(float(a - b)) for q, a, b in zip(qids, ly, ln)}                               # Yes の対数オッズ

    def choice_probs(labels, qid):
        text = (CHOICE_PREFIX + "Candidate genes:\n" + "\n".join(f"{i + 1}. {g}" for i, g in enumerate(labels)) +
                f"\n{len(labels) + 1}. None of the above genes seem clearly relevant\n" +
                f"Question: {CHOICE_QUESTIONS[qid].format(disease=D['name'])}" + ("" if CHAT else " Answer: "))
        x = gtok([wrap(text)], return_tensors="pt", add_special_tokens=False).to(GDEV)
        with torch.no_grad():
            lg = gmodel(**x, logits_to_keep=1).logits[0, -1, :].float().cpu().numpy()
        l = np.array([lg[DIGIT[n]] for n in range(1, GROUP_SIZE + 2)]); p = np.exp(l - l.max())   # 数字1〜6だけを取り出して正規化
        return p / p.sum()
    MODEL_NAME = M["name"]

RUNLOG["model"] = {"use": MODEL_USE, "backend": M["backend"], "model": MODEL_NAME, "prompt_wrap": M["prompt_wrap"]}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
# --- 動作確認：TNF で両順の対数オッズを出す ---
t0 = time.time()
print(f"model: {MODEL_NAME}（{MODEL_USE} / {M['backend']} / prompt_wrap={M['prompt_wrap']}）| test TNF:",
      {o: round(yes_logodds(G.at["TNF", "gene_label"], "TNF", ["M3s"], o)["M3s"], 2) for o in ("yes_first", "no_first")}, f"| {time.time() - t0:.1f}s")

## 段階2：粗い採点

### このセルがすること：段階1で残した遺伝子に、質問（M3s）を「Yes or No」の順だけで聞く
- 無作為に選んだ `stage2.calib_genes` 個だけは「No or Yes」の順でも聞き、順序の偏り δ（対数オッズの差の平均）を推定します。残りは `yes_first − δ/2` で補正します（両順の平均に相当）。
- `checkpoint_every` 件ごとに `stage2.csv` へ保存し、再実行すると続きから再開します。最後に上位 `stage2.keep` 件に印（`keep2`）を付けます。

In [ ]:
c = cfg["stage2"]; f2 = os.path.join(OUT, "stage2.csv"); q = c["question"]
cand = s1[s1["keep1"]]["symbol"].tolist()                               # 段階1で残した遺伝子
# --- 途中まで終わっていれば、その結果を読んで続きから ---
rows = pd.read_csv(f2)[["symbol", "lo_yes_first", "lo_no_first"]].to_dict("records") if os.path.exists(f2) else []
done = {r["symbol"] for r in rows}
calib = set(random.Random(c["seed"]).sample(cand, min(c["calib_genes"], len(cand))))   # 両順で聞く遺伝子（無作為）
todo = [s for s in cand if s not in done]; t0 = time.time()
print(f"段階2: {len(cand)} 件（済み {len(done)}、残り {len(todo)}、両順で聞く {len(calib)} 件）")
# --- 1遺伝子ずつ「Yes or No」の順で聞く（calib の遺伝子だけ「No or Yes」でも聞く） ---
for k, s in enumerate(todo, 1):
    yf = yes_logodds(G.at[s, "gene_label"], s, [q], "yes_first")[q]
    nf = yes_logodds(G.at[s, "gene_label"], s, [q], "no_first")[q] if s in calib else np.nan
    rows.append({"symbol": s, "lo_yes_first": yf, "lo_no_first": nf})
    if k % cfg["checkpoint_every"] == 0 or k == len(todo):
        pd.DataFrame(rows).to_csv(f2, index=False)
        print(f"  {k}/{len(todo)}（{(time.time() - t0) / k:.2f} 秒/遺伝子、残り約 {(len(todo) - k) * (time.time() - t0) / k / 60:.0f} 分）")
# --- 順序の偏り δ を、両順で聞いた遺伝子の差の平均として推定し、1順だけの遺伝子を補正する ---
s2 = pd.DataFrame(rows)
both = s2.dropna(subset=["lo_no_first"])
delta = float((both["lo_yes_first"] - both["lo_no_first"]).mean()) if len(both) else 0.0
s2["score2"] = np.where(s2["lo_no_first"].notna(), (s2["lo_yes_first"] + s2["lo_no_first"]) / 2, s2["lo_yes_first"] - delta / 2)
# --- 点数の高い順に並べ、上位 stage2.keep 件に印を付ける ---
s2 = s2.sort_values("score2", ascending=False).reset_index(drop=True)
s2["rank2"] = np.arange(1, len(s2) + 1); s2["keep2"] = s2["rank2"] <= c["keep"]; s2["delta"] = delta
s2.to_csv(f2, index=False)
RUNLOG["stage2"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "seconds_this_session": round(time.time() - t0), "input": len(cand), "kept": int(s2["keep2"].sum()),
                    "delta_logodds": round(delta, 4), "calib_genes": len(calib)}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(f"段階2: 順序の偏り δ = {delta:+.2f}（対数オッズ）→ {int(s2['keep2'].sum())} 件")
s2.head(15)

### 図2：段階2の点数の分布
### このセルがすること：粗い採点の分布と、残す線（`stage2.keep` 件目）を描き、既知の標的の位置と順序の偏り δ を示す

In [ ]:
cut = s2.loc[s2["keep2"], "score2"].min()
kn = s2[s2["symbol"].isin(KNOWN)]
fig = go.Figure(go.Histogram(x=s2["score2"], nbinsx=80, marker_color="#c9c8c2", name="段階1で残した遺伝子"))
fig.add_vline(x=cut, line=dict(color="#0b0b0b", dash="dot"), annotation_text=f"上位 {cfg['stage2']['keep']} 件の線")
fig.add_trace(go.Scatter(x=kn["score2"], y=[0] * len(kn), mode="markers+text", text=kn["symbol"], textposition="top center", name="既知の標的",
                         marker=dict(color=np.where(kn["keep2"], "#2a78d6", "#d64545"), size=10, symbol="triangle-up"),
                         customdata=np.c_[kn["symbol"], kn["rank2"]], hovertemplate="<b>%{customdata[0]}</b><br>点数 %{x:.2f}（%{customdata[1]}位）<extra></extra>"))
fig.update_layout(**PLOT, height=420, xaxis_title="粗い採点（補正後の対数オッズ）", yaxis_title="遺伝子の数",
                  title=f"段階2：粗い採点の分布（順序の偏り δ = {s2['delta'].iloc[0]:+.2f}。▲青＝残った既知、▲赤＝ここで落ちた既知）")
fig.show(); name = "02_stage2_scores.html"; CHART_TITLES[name] = "段階2：粗い採点の分布と線"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

## 段階3：精密な採点（最終順位）

### このセルがすること：段階2で残した遺伝子に、M3s を両順（`yes_first` は段階2の値を再利用）と、機能情報つきの M3sF を両順で聞き、対数オッズを平均する
- 機能情報は段階0の後で作った `FUNC_TEXT`（この疾患の病名を含む文は除いてある）を使います。
- `m3s`・`m3sf`：それぞれ両順の平均。`score`：その2つの平均＝**最終順位**。上位 `stage3.keep` 件がトップ500です。

In [ ]:
c = cfg["stage3"]; f3 = os.path.join(OUT, "stage3.csv"); q = c["question"]
cand = s2[s2["keep2"]]                                                  # 段階2で残した遺伝子
rows = pd.read_csv(f3)[["symbol", "m3s", "m3sf", "has_function"]].to_dict("records") if os.path.exists(f3) else []
done = {r["symbol"] for r in rows}; todo = cand[~cand["symbol"].isin(done)]; t0 = time.time()
print(f"段階3: {len(cand)} 件（済み {len(done)}、残り {len(todo)}）")
# --- 1遺伝子ずつ：M3s の No/Yes 順（未実施なら）＋ 機能情報つき M3sF の両順を聞く ---
for k, r in enumerate(todo.itertuples(), 1):
    s, lab = r.symbol, G.at[r.symbol, "gene_label"]
    extra = FUNC_TEXT[s] if HAS_FUNC[s] else ""                          # 機能情報の3行（記載が無ければ入れない）
    nf = r.lo_no_first if not pd.isna(r.lo_no_first) else yes_logodds(lab, s, [q], "no_first")[q]
    fy = yes_logodds(lab, s, [q], "yes_first", extra)[q]; fn = yes_logodds(lab, s, [q], "no_first", extra)[q]
    rows.append({"symbol": s, "m3s": (r.lo_yes_first + nf) / 2, "m3sf": (fy + fn) / 2, "has_function": bool(extra)})
    if k % cfg["checkpoint_every"] == 0 or k == len(todo):
        pd.DataFrame(rows).to_csv(f3, index=False)
        print(f"  {k}/{len(todo)}（{(time.time() - t0) / k:.2f} 秒/遺伝子、残り約 {(len(todo) - k) * (time.time() - t0) / k / 60:.0f} 分）")
# --- 最終の点数＝ M3s と M3sF（どちらも両順の平均）の平均。高い順が最終順位 ---
s3 = pd.DataFrame(rows); s3["score"] = (s3["m3s"] + s3["m3sf"]) / 2
s3 = s3.sort_values("score", ascending=False).reset_index(drop=True); s3["rank"] = np.arange(1, len(s3) + 1)
s3.to_csv(f3, index=False)
RUNLOG["stage3"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "seconds_this_session": round(time.time() - t0), "input": len(cand), "kept": c["keep"],
                    "with_function_info": int(s3["has_function"].sum())}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(f"段階3: 上位 {c['keep']} 件が最終順位")
s3.head(15)

### 図3：段階3の M3s × M3sF
### このセルがすること：段階3の全件を、横＝M3s（機能情報なし）、縦＝M3sF（機能情報あり）に描き、トップ500と既知の標的を色分けする
- 右上ほど最終の点数が高い。点線より上は機能情報で点数が上がった遺伝子です。

In [ ]:
in_top = s3["rank"] <= cfg["stage3"]["keep"]; is_kn = s3["symbol"].isin(KNOWN)
fig = go.Figure()
for mask, name_, col, size in ((~in_top, "トップ500の外", "#c9c8c2", 5), (in_top & ~is_kn, "トップ500", "#eb6834", 6), (is_kn, "既知の標的", "#2a78d6", 10)):
    d = s3[mask]
    fig.add_trace(go.Scatter(x=d["m3s"], y=d["m3sf"], mode="markers", name=name_, marker=dict(color=col, size=size, opacity=0.8),
                             customdata=np.c_[d["symbol"], d["rank"]], hovertemplate="<b>%{customdata[0]}</b>（%{customdata[1]}位）<br>M3s %{x:.2f}<br>M3sF %{y:.2f}<extra></extra>"))
lo, hi = float(s3[["m3s", "m3sf"]].min().min()), float(s3[["m3s", "m3sf"]].max().max())
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color="#0b0b0b", dash="dot"), showlegend=False, hoverinfo="skip"))
fig.update_layout(**PLOT, height=600, xaxis_title="M3s（対数オッズ）", yaxis_title="M3sF（対数オッズ）", title="段階3：機能情報なし × あり（最終の点数＝2つの平均）")
fig.show(); name = "03_stage3_m3s_m3sf.html"; CHART_TITLES[name] = "段階3：M3s × M3sF"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

## 段階4：上位の並べ直し（確認用）

### `luce(groups, n)` — Bradley-Terry（Luce の選択モデル）の強さを推定する
どんな def か：groups は（メンバーの番号、各メンバーが選ばれた確率）の並び。遺伝子 i がグループから選ばれる確率を π_i ÷（グループ内の π の合計）と置き、全グループの結果に合う π を MM 法で反復推定します。全勝・全敗でも発散しないよう、各項目に「強さ1の仮想相手との 0.5 勝 0.5 敗」を足します。スイス式では毎ラウンド推定し直すので関数にしています。
return：log π（numpy 配列、幾何平均が1）。

### このセルがすること：段階3の上位 `stage4.top` 件を、5択（M2r）× スイス式 × Bradley-Terry で並べ直す
- 最初の `random_rounds` ラウンドはランダムに5つずつ組み、以降は「それまでの強さ＋小さな乱数」の順に並べて5つずつ組みます（強さの近い遺伝子どうし）。組の中の並び（番号）は毎回シャッフルします。
- `rank4`：段階4での順位。最終順位は変えず、段の判定（A）と参考にだけ使います。

In [ ]:
def luce(groups, n, iters=1000, prior=0.5):
    W, pi = np.full(n, prior), np.ones(n)                               # W：やわらかい勝ち数（選ばれた確率の合計）、pi：強さ
    for mem, p in groups:
        for m, x in zip(mem, p): W[m] += x
    for _ in range(iters):
        den = 2 * prior / (pi + 1)                                       # 仮想相手（強さ1）との対戦の分
        for mem, p in groups: den[mem] += 1 / pi[mem].sum()
        new = W / den; new /= np.exp(np.mean(np.log(new)))               # MM 法の更新式。幾何平均を1にそろえる
        if np.max(np.abs(np.log(new) - np.log(pi))) < 1e-10: pi = new; break
        pi = new
    return np.log(pi)

# --- 段階3の上位 stage4.top 件と「該当なし」に番号を振る ---
c = cfg["stage4"]; top = s3.head(c["top"])["symbol"].tolist(); ix = {s: k for k, s in enumerate(top + ["NONE"])}
rng, rows, t0 = random.Random(c["seed"]), [], time.time()
f4r = os.path.join(OUT, "stage4_rounds.csv")
for rnd in (range(c["rounds"]) if not os.path.exists(f4r) else []):   # 保存済みなら飛ばして読む（再開用）
    if rnd < c["random_rounds"]:                                              # ランダムに組む
        order = top[:]; rng.shuffle(order); groups = []
        for start in range(0, len(order), GROUP_SIZE):
            chunk = order[start:start + GROUP_SIZE]
            if len(chunk) < GROUP_SIZE: chunk = chunk + rng.sample([s for s in top if s not in chunk], GROUP_SIZE - len(chunk))
            groups.append(chunk)
    else:                                                                     # スイス式：強さの近いものどうしで組む
        L = pd.DataFrame(rows)
        st = luce([([ix[s] for s in d.sort_values("position")["symbol"]] + [ix["NONE"]],
                    np.r_[d.sort_values("position")["p"].values, d["p_none"].iloc[0]]) for _, d in L.groupby(["round", "group"])], len(ix))
        sv = np.array([st[ix[s]] for s in top]); sv = sv + np.array([rng.gauss(0, 1) for _ in top]) * c["jitter"] * (sv.std() or 1)
        order = [top[k] for k in np.argsort(-sv)]; groups = []
        for start in range(0, len(order), GROUP_SIZE):
            chunk = order[start:start + GROUP_SIZE]
            if len(chunk) < GROUP_SIZE: chunk = chunk + order[start - (GROUP_SIZE - len(chunk)):start]
            chunk = chunk[:]; rng.shuffle(chunk); groups.append(chunk)
    # 各グループに5択で聞き、各遺伝子が選ばれた確率を記録
    for gi, chunk in enumerate(groups):
        p = choice_probs([G.at[s, "gene_label"] for s in chunk], c["question"])
        for slot, s in enumerate(chunk):
            rows.append({"round": rnd, "group": gi, "position": slot + 1, "symbol": s, "p": round(float(p[slot]), 6), "p_none": round(float(p[-1]), 6)})
    print(f"  round {rnd + 1}/{c['rounds']}（{'random' if rnd < c['random_rounds'] else 'swiss'}、{time.time() - t0:.0f}s）")
# --- 全ラウンドの結果で強さを推定し、「該当なし」より強いか・順位を出す ---
L = pd.DataFrame(rows) if rows else pd.read_csv(f4r); L.to_csv(f4r, index=False)
st = luce([([ix[s] for s in d.sort_values("position")["symbol"]] + [ix["NONE"]], np.r_[d.sort_values("position")["p"].values, d["p_none"].iloc[0]])
           for _, d in L.groupby(["round", "group"])], len(ix))
s4 = pd.DataFrame({"symbol": top, "bt": [st[ix[s]] for s in top]}); s4["bt_above_none"] = s4["bt"] > st[ix["NONE"]]
s4["rank4"] = s4["bt"].rank(ascending=False, method="first").astype(int)
s4.to_csv(os.path.join(OUT, "stage4.csv"), index=False)
RUNLOG["stage4"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "seconds": round(time.time() - t0), "genes": len(top), "groups": int(L.groupby(["round", "group"]).ngroups),
                    "rank_corr_with_stage3": round(float(s3.head(len(top)).merge(s4, on="symbol")[["rank", "rank4"]].corr(method="spearman").iloc[0, 1]), 3)}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(f"段階4: 上位 {len(top)} 件 | 段階3と段階4の順位相関 {s3.head(len(top)).merge(s4, on='symbol')[['rank', 'rank4']].corr(method='spearman').iloc[0, 1]:.2f}")

### 図4：段階3の順位 × 段階4の順位
### このセルがすること：上位の遺伝子を、横＝段階3の順位、縦＝段階4（5択・スイス式）の順位に描く。左下ほど両方で上位

In [ ]:
d = s3.merge(s4, on="symbol"); d["known"] = d["symbol"].isin(KNOWN)
fig = go.Figure()
for kflag, name_, col in ((False, "その他", "#eb6834"), (True, "既知の標的", "#2a78d6")):
    x = d[d["known"] == kflag]
    fig.add_trace(go.Scatter(x=x["rank"], y=x["rank4"], mode="markers+text", text=x["symbol"], textposition="top center", textfont=dict(size=8), name=name_,
                             marker=dict(color=col, size=8), hovertemplate="%{text}<br>段階3 %{x}位<br>段階4 %{y}位<extra></extra>"))
fig.add_trace(go.Scatter(x=[1, len(d)], y=[1, len(d)], mode="lines", line=dict(color="#c3c2b7", dash="dot"), showlegend=False, hoverinfo="skip"))
fig.update_layout(**PLOT, height=650, xaxis=dict(title="段階3の順位", autorange="reversed"), yaxis=dict(title="段階4の順位", autorange="reversed"),
                  title="段階4：2つの聞き方の順位の比較（左下＝両方で上位＝段 A の候補）")
fig.show(); name = "04_stage4_rerank.html"; CHART_TITLES[name] = "段階4：段階3と段階4の順位"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

## 段階5：理由のラベル

### このセルがすること：トップ500に H3（直接効く）・R1（代償経路）・B1（機構が一致）を両順で聞き、500件の中で特に強い条件をラベルにする
- 上位の遺伝子は H3・R1 がほぼ全員 Yes、B1 はほぼ全員 No になるので、Yes/No のままでは区別が付きません。各条件の対数オッズを500件の中で標準化し、平均より `stage5.z_threshold` 標準偏差以上高い条件をラベルにします（無ければ最も高い条件）。
- 機能情報は入れずに聞きます（検証と同じ）。順位には使いません。

In [ ]:
c = cfg["stage5"]; f5 = os.path.join(OUT, "stage5.csv"); qs = list(c["questions"])
target = s3.head(cfg["stage3"]["keep"])["symbol"].tolist()
rows = pd.read_csv(f5)[["symbol"] + qs].to_dict("records") if os.path.exists(f5) else []
done = {r["symbol"] for r in rows}; todo = [s for s in target if s not in done]; t0 = time.time()
# --- 1遺伝子ずつ H3・R1・B1 を両順で聞き、対数オッズを平均 ---
for k, s in enumerate(todo, 1):
    a = yes_logodds(G.at[s, "gene_label"], s, qs, "yes_first"); b = yes_logodds(G.at[s, "gene_label"], s, qs, "no_first")
    rows.append({"symbol": s, **{x: (a[x] + b[x]) / 2 for x in qs}})
    if k % cfg["checkpoint_every"] == 0 or k == len(todo):
        pd.DataFrame(rows).to_csv(f5, index=False); print(f"  {k}/{len(todo)}（{time.time() - t0:.0f}s）")
# --- 条件ごとに500件の中で標準化し、z_threshold 以上の条件をラベルにする（無ければ最大の条件） ---
s5 = pd.DataFrame(rows); z = (s5[qs] - s5[qs].mean()) / s5[qs].std()
s5["reasons"] = ["・".join(c["questions"][x] for x in qs if z.at[i, x] >= c["z_threshold"]) or c["questions"][z.loc[i].idxmax()] for i in s5.index]
s5.to_csv(f5, index=False)
RUNLOG["stage5"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "seconds_this_session": round(time.time() - t0), "genes": len(s5),
                    "labels": {k: int(v) for k, v in s5["reasons"].value_counts().head(10).items()}}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
s5["reasons"].value_counts().head(10)

### 図5：理由のラベル
### このセルがすること：理由のラベルの内訳と、H3・R1・B1 それぞれの対数オッズの分布を描く

In [ ]:
qs = list(cfg["stage5"]["questions"])
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], subplot_titles=("理由のラベルの内訳（上位10種）", "条件ごとの対数オッズ"))
vc = s5["reasons"].value_counts().head(10)
fig.add_trace(go.Bar(x=vc.values, y=vc.index, orientation="h", marker_color="#2a78d6", showlegend=False), row=1, col=1)
for q in qs:
    fig.add_trace(go.Box(y=s5[q], name=f"{q} {cfg['stage5']['questions'][q]}", boxpoints="all", jitter=0.4, pointpos=0, marker=dict(size=3),
                         customdata=s5["symbol"], hovertemplate="<b>%{customdata}</b><br>%{y:.2f}<extra></extra>"), row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_layout(**PLOT, height=480, showlegend=False, title="段階5：理由のラベル（500件の中で相対的に強い条件）")
fig.show(); name = "05_stage5_reasons.html"; CHART_TITLES[name] = "段階5：理由のラベル"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

## 段階6：裏付けの注釈（順位を付けたあと）

### このセルがすること：Open Targets から、この疾患で臨床段階の薬がある標的（最も進んだ段階と薬の名前）と、関連スコアを取得する
- 疾患 ID は病名で検索します（見つかった ID と名前を表示し、`run_log.yaml` に記録。名前が病名と違えば警告）。臨床段階は ChEMBL 由来です。順位付けには使いません。

In [ ]:
if cfg["stage6"]["enabled"]:
    api = cfg["stage6"]["api"]
    # --- 疾患 ID を病名で検索する（一番上の候補を使う） ---
    req = urllib.request.Request(api, headers={"Content-Type": "application/json"}, data=json.dumps({
        "query": 'query($q:String!){search(queryString:$q,entityNames:["disease"],page:{index:0,size:1}){hits{id name}}}', "variables": {"q": D["name"]}}).encode())
    with urllib.request.urlopen(req, timeout=120) as r: hit = json.load(r)["data"]["search"]["hits"][0]
    did = hit["id"]
    print(f"Open Targets: {did}（{hit['name']}）" + ("" if hit["name"].lower() == D["name"].lower() else "  [警告] 病名と名前が違います。確認してください"))
    # --- この疾患の薬と臨床試験中の薬（最も進んだ段階と、作用する標的） ---
    req = urllib.request.Request(api, headers={"Content-Type": "application/json"}, data=json.dumps({
        "query": "query($id:String!){disease(efoId:$id){drugAndClinicalCandidates{rows{maxClinicalStage drug{name mechanismsOfAction{rows{targets{approvedSymbol}}}}}}}}",
        "variables": {"id": did}}).encode())
    with urllib.request.urlopen(req, timeout=120) as r: rows = json.load(r)["data"]["disease"]["drugAndClinicalCandidates"]["rows"]
    best, drugs = {}, {}                                                  # 標的ごとの最も進んだ臨床段階と、薬の名前
    rank_of = lambda st: STAGE_ORDER.index(st) if st in STAGE_ORDER else -1               # UNKNOWN などは最も低い
    for r in rows:
        for m in ((r["drug"] or {}).get("mechanismsOfAction") or {}).get("rows", []):
            for t in m["targets"]:
                s = t["approvedSymbol"]
                if s not in best or rank_of(r["maxClinicalStage"]) > rank_of(best[s]): best[s] = r["maxClinicalStage"]
                drugs.setdefault(s, set()).add((r["drug"] or {}).get("name", ""))
    assoc, page = {}, 0
    while True:                                                               # 関連スコア（500件ずつ）
        req = urllib.request.Request(api, headers={"Content-Type": "application/json"}, data=json.dumps({
            "query": "query($id:String!,$i:Int!){disease(efoId:$id){associatedTargets(page:{index:$i,size:500}){count rows{score target{approvedSymbol}}}}}",
            "variables": {"id": did, "i": page}}).encode())
        with urllib.request.urlopen(req, timeout=120) as r: x = json.load(r)["data"]["disease"]["associatedTargets"]
        for row in x["rows"]: assoc[row["target"]["approvedSymbol"]] = row["score"]
        page += 1
        if page * 500 >= x["count"] or not x["rows"]: break
    # --- 標的ごとの表にして保存 ---
    s6 = pd.DataFrame({"symbol": sorted(set(best) | set(assoc))})
    s6["ot_max_stage"] = s6["symbol"].map(best); s6["ot_drugs"] = s6["symbol"].map(lambda s: "; ".join(sorted(drugs.get(s, set())))[:200])
    s6["ot_assoc_score"] = s6["symbol"].map(assoc); s6["ot_disease_id"] = did
    s6.to_csv(os.path.join(OUT, "stage6.csv"), index=False)
    RUNLOG["stage6"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "open_targets_id": did, "open_targets_name": hit["name"], "targets_with_clinical_drug": len(best), "targets_with_association": len(assoc)}
    yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    print(f"段階6: Open Targets（{did}）臨床段階の薬がある標的 {len(best)}、関連スコア {len(assoc)}")
    display(s6["ot_max_stage"].value_counts())

## 最終ランキング（トップ500）

### このセルがすること：各段階の結果をまとめ、確からしさの段（A / B / C）と裏付けを付けて `top500.csv` に書く
- **A**：1〜`tiers.A` 位で、段階4でも `tiers.A` 位以内（2つの聞き方で一致）。**B**：〜`tiers.B` 位。**C**：〜500位。
- `evidence`：承認薬あり／臨床段階の薬あり／なし（新規候補）。

In [ ]:
# --- 段階3の上位500件に、各段階の結果をつなげる ---
top = s3.head(cfg["stage3"]["keep"]).copy()
top = top.merge(s2[["symbol", "rank2", "score2"]], on="symbol", how="left").merge(s1[["symbol", "rank1", "score1"]], on="symbol", how="left")
top = top.merge(s4[["symbol", "rank4", "bt", "bt_above_none"]], on="symbol", how="left").merge(s5, on="symbol", how="left")
if cfg["stage6"]["enabled"]: top = top.merge(s6[["symbol", "ot_max_stage", "ot_drugs", "ot_assoc_score"]], on="symbol", how="left")
top.insert(2, "gene_name", top["symbol"].map(G["gene_name"]))
top["function"] = [FUNC_TEXT[s].splitlines()[0][len("Function: "):][:160] for s in top["symbol"]]
# --- 確からしさの段と裏付けの区分を付ける ---
A, B_ = cfg["tiers"]["A"], cfg["tiers"]["B"]
top["tier"] = np.where((top["rank"] <= A) & (top["rank4"] <= A), "A", np.where(top["rank"] <= B_, "B", "C"))
if "ot_max_stage" in top:
    top["evidence"] = np.where(top["ot_max_stage"].isin(["APPROVAL", "PREAPPROVAL"]), "承認薬あり",
                               np.where(top["ot_max_stage"].notna(), "臨床段階の薬あり", "なし（新規候補）"))
top.to_csv(os.path.join(OUT, "top500.csv"), index=False)
RUNLOG["final"] = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "genes": len(top), "tiers": {k: int(v) for k, v in top["tier"].value_counts().items()},
                   "evidence": {k: int(v) for k, v in top["evidence"].value_counts().items()} if "evidence" in top else None}
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(top["tier"].value_counts().to_dict(), "|", top["evidence"].value_counts().to_dict() if "evidence" in top else "", "→", os.path.join(OUT, "top500.csv"))
top.head(30)[["rank", "symbol", "gene_name", "tier", "score", "rank4", "reasons"] + (["evidence", "ot_max_stage"] if "evidence" in top else [])]

## 評価

### このセルがすること：既知の標的が各段階でどれだけ残ったかを出す（順位付けには使わない）
- 正解は2種類：`evaluation.known_files` の既知標的（ある疾患だけ）と、Open Targets で PHASE_2 以上の薬がある標的。
- `recall@K`：正解のうち上位 K 件に入った割合。`濃縮率`：上位 K 件に正解が入る割合が、ランダムに選んだ場合の何倍か。

In [ ]:
# --- 正解（既知の標的）を集める：known_file と Open Targets の PHASE_2 以上 ---
pos = {}
if KNOWN: pos["既知（known_file）"] = KNOWN                                # evaluation.known_files に一覧がある疾患だけ
if cfg["stage6"]["enabled"]:
    pos["Open Targets（PHASE_2 以上）"] = set(s6.loc[s6["ot_max_stage"].isin(STAGE_ORDER[STAGE_ORDER.index("PHASE_2"):]), "symbol"])
# --- 段階ごとの順位で、上位 K 件に正解がいくつ入ったか（recall@K）とランダム比（濃縮率）を数える ---
N = len(U); rows = []
for pname, Pset in pos.items():
    Pset = Pset & set(U["symbol"])
    for sname, ranks in (("段階1（リランカー）", s1.set_index("symbol")["rank1"]), ("段階2（粗い採点）", s2.set_index("symbol")["rank2"]), ("段階3（最終）", s3.set_index("symbol")["rank"])):
        for K in (50, 200, 500, 2000, 5000):
            hit = sum(1 for s in Pset if s in ranks.index and ranks[s] <= K)
            rows.append({"正解": pname, "正解の数": len(Pset), "段階": sname, "K": K, "上位K件に入った数": hit,
                         "recall@K": hit / len(Pset) if Pset else np.nan, "濃縮率（ランダム比）": (hit / K) / (len(Pset) / N) if Pset else np.nan})
ev = pd.DataFrame(rows); ev.to_csv(os.path.join(OUT, "evaluation.csv"), index=False)
if len(ev):
    RUNLOG["evaluation_recall"] = {f"{r['正解']}｜{r['段階']}｜@{r['K']}": round(float(r["recall@K"]), 3) for _, r in ev[ev["K"].isin([500, 2000, 5000])].iterrows()}
    yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
ev.round(3)

## 最後の図（ホバーで遺伝子名）

### 図6：絞り込みの過程
### このセルがすること：各段階で残した件数と、そこに正解（既知の標的・Open Targets の PHASE_2 以上）が何件残ったかを描く

In [ ]:
steps = [("全遺伝子", set(U["symbol"])), ("段階1", set(s1.loc[s1["keep1"], "symbol"])), ("段階2", set(s2.loc[s2["keep2"], "symbol"])),
         ("段階3（トップ500）", set(s3.loc[s3["rank"] <= cfg["stage3"]["keep"], "symbol"])), (f"段 A（{cfg['tiers']['A']} 位以内）", set(top.loc[top["tier"] == "A", "symbol"]))]
fig = make_subplots(rows=1, cols=2, subplot_titles=("残した件数（対数目盛）", "正解のうち残った割合"))
fig.add_trace(go.Bar(x=[a for a, _ in steps], y=[len(b) for _, b in steps], marker_color="#9ec0ec", text=[len(b) for _, b in steps], textposition="outside", showlegend=False), row=1, col=1)
for pname, Pset in pos.items():
    Pset = Pset & set(U["symbol"])
    fig.add_trace(go.Scatter(x=[a for a, _ in steps], y=[len(Pset & b) / len(Pset) for _, b in steps], mode="lines+markers+text", name=f"{pname}（{len(Pset)}）",
                             text=[f"{len(Pset & b)}" for _, b in steps], textposition="top center"), row=1, col=2)
fig.update_yaxes(type="log", row=1, col=1); fig.update_yaxes(range=[0, 1.1], row=1, col=2)
fig.update_layout(**PLOT, height=450, title=f"{D['name']}: 絞り込みの過程（右：数字＝残った正解の数）")
fig.show(); name = "06_funnel.html"; CHART_TITLES[name] = "絞り込みの過程（件数と正解の残り方）"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

### 図7：評価の曲線
### このセルがすること：段階ごとの順位で、上位 K 件に正解が入った割合（recall@K）を描く（ホバーで、ランダムと比べて何倍か）

In [ ]:
fig = go.Figure()
for (pname, sname), d in ev.groupby(["正解", "段階"]):
    fig.add_trace(go.Scatter(x=d["K"], y=d["recall@K"], mode="lines+markers", name=f"{pname}｜{sname}",
                             customdata=np.c_[d["上位K件に入った数"], d["正解の数"], d["濃縮率（ランダム比）"]],
                             hovertemplate="%{fullData.name}<br>上位 %{x} 件: %{customdata[0]} / %{customdata[1]}（recall %{y:.2f}、濃縮 %{customdata[2]:.1f} 倍）<extra></extra>"))
fig.update_layout(**PLOT, height=450, xaxis=dict(title="K（上位の件数）", type="log"), yaxis=dict(title="recall@K", range=[0, 1.02]),
                  title=f"{D['name']}: 正解のうち上位 K 件に入った割合（段階ごと）")
fig.show(); name = "07_recall.html"; CHART_TITLES[name] = "評価：recall@K と濃縮率"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

### 図8：トップ500
### このセルがすること：トップ500を、横＝M3s、縦＝M3sF に描き、裏付け（承認薬・臨床段階・新規候補）で色分け、段（A>B>C）を点の大きさで示す

In [ ]:
EVC = {"承認薬あり": "#2a78d6", "臨床段階の薬あり": "#8a5cd1", "なし（新規候補）": "#eb6834"}
evid = top["evidence"] if "evidence" in top else pd.Series("なし（新規候補）", index=top.index)
fig = go.Figure()
for name_, col in EVC.items():
    d = top[evid == name_]
    fig.add_trace(go.Scatter(x=d["m3s"], y=d["m3sf"], mode="markers", name=name_,
                             marker=dict(color=col, size=np.where(d["tier"] == "A", 11, np.where(d["tier"] == "B", 8, 6)), opacity=0.8, line=dict(width=1, color="white")),
                             customdata=np.c_[d["symbol"], d["rank"], d["tier"], d["reasons"].fillna(""), d["function"].fillna("")],
                             hovertemplate="<b>%{customdata[0]}</b>（%{customdata[1]}位・段 %{customdata[2]}）<br>%{customdata[3]}<br>%{customdata[4]}<extra></extra>"))
fig.update_layout(**PLOT, height=600, xaxis_title="M3s（対数オッズ）", yaxis_title="M3sF（対数オッズ）",
                  title=f"{D['name']}: トップ500（色＝裏付け、点の大きさ＝段 A>B>C）")
fig.show(); name = "08_top500.html"; CHART_TITLES[name] = "トップ500（裏付けと段）"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")

### 図9：トップ500の内訳と、全図の一覧ページ
### このセルがすること：段・理由・裏付けの内訳を描き、ここまでの全図へのリンクを並べた `charts/index.html` を作る

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("確からしさの段", "理由のラベル（上位10種）", "裏付け（Open Targets）"))
for j, col in enumerate(("tier", "reasons", "evidence"), start=1):
    if col in top:
        vc = top[col].fillna("").value_counts().head(10)
        fig.add_trace(go.Bar(x=vc.index, y=vc.values, showlegend=False, marker_color="#2a78d6"), row=1, col=j)
fig.update_layout(**PLOT, height=420, title="トップ500の内訳")
fig.show(); name = "09_breakdown.html"; CHART_TITLES[name] = "トップ500の内訳（段・理由・裏付け）"
fig.write_html(os.path.join(CHARTS, name), include_plotlyjs="directory")
# --- 一覧ページ：実行の条件の要約と、全図へのリンク ---
links = "".join(f"<li><a href='{f}'>{t}</a> <small>({f})</small></li>" for f, t in sorted(CHART_TITLES.items()))
info = (f"<p>疾患：{D['name']}（{DISEASE_KEY}）｜実行：{RUN_ID}｜全遺伝子 {len(U)} → 段階1 {cfg['stage1']['keep']} → 段階2 {cfg['stage2']['keep']} → "
        f"トップ {cfg['stage3']['keep']}｜条件は <code>../conditions/</code>、記録は <code>../run_log.yaml</code></p>")
open(os.path.join(CHARTS, "index.html"), "w", encoding="utf-8").write(
    f"<html><head><meta charset='utf-8'><title>{D['name']} {RUN_ID}</title></head><body style='font-family:sans-serif;max-width:900px;margin:auto'>"
    f"<h2>{D['name']}：トップ{cfg['stage3']['keep']} の図の一覧</h2>{info}<ol>{links}</ol></body></html>")
RUNLOG["charts"] = sorted(CHART_TITLES)
yaml.safe_dump(RUNLOG, open(RUNLOG_PATH, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print("図の一覧:", os.path.join(CHARTS, "index.html"))